# Edge IIoT - Binary Classification


In [13]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [14]:
import torch

print(torch.__version__) # X.XX.X+cuXXX / If X.XX.X+cpu it won't work
print(torch.cuda.is_available()) # False
print(torch.version.cuda) # None or mismatched version
print(torch.cuda.device_count()) # 0

2.10.0+cu128
True
12.8
1


In [15]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from src.config import DATASETS, SEED

# Seleccionamos el dataset a analizar
dataset_name = "edge_iiot"
config = DATASETS[dataset_name]

print("\n--- Dataset Information ---")
print(f"Name: {dataset_name.upper()}")
print(f"Path: {config['processed_path']}\n")

filename = "ML-EdgeIIoT-dataset"

csv_path = config['processed_path'] / f"{filename}.pkl"
print(f"CSV Path: {csv_path}\n")

print("Loading dataset... (This may take a while)")
df = pd.read_pickle(csv_path)

print(f"\nDataset loaded with shape: {df.shape}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload

--- Dataset Information ---
Name: EDGE_IIOT
Path: /home/uo294319/ML-NIDS-IIoT/data/processed/edge_iiot

CSV Path: /home/uo294319/ML-NIDS-IIoT/data/processed/edge_iiot/ML-EdgeIIoT-dataset.pkl

Loading dataset... (This may take a while)

Dataset loaded with shape: (150005, 59)


## 1. Undersampling and Balancing

In [16]:
target     = 'Attack_label'
target_str = 'Attack_type'

In [17]:
y_str = df[target_str]
df_no_label = df.drop(columns=[target_str])

In [18]:
MAX_PRESENCE = 0.02

counts = y_str.value_counts()
total_rows = len(y_str)

print(counts)
print("\nTOTAL: ", total_rows)

Attack_type
Normal                   24231
DDoS_UDP                 14498
DDoS_ICMP                13306
DDoS_HTTP                10560
SQL_injection            10297
Uploading                10260
Ransomware               10196
Vulnerability_scanner    10065
Backdoor                  9992
Password                  9980
Port_Scanning             9690
XSS                       9620
DDoS_TCP                  6011
Fingerprinting             932
MITM                       367
Name: count, dtype: int64

TOTAL:  150005


In [19]:
print(f"{'Attack Type':<25} | {'Old Count':<15} | {'New Count':<15}\n" + "-"*55)

sampling_strategy = {}
for attack_type, count in counts.items():
    current_presence = count / total_rows

    if current_presence > MAX_PRESENCE:
        new_count = int(total_rows * MAX_PRESENCE)
    else:
        new_count = count

    sampling_strategy[attack_type] = new_count
    print(f"{attack_type:<25} | {count:<15} | {new_count:<15}")


print("-"*55 + f"\n{'TOTAL':<25} | {counts.sum():<15} | {sum(sampling_strategy.values()):<15}")


Attack Type               | Old Count       | New Count      
-------------------------------------------------------
Normal                    | 24231           | 3000           
DDoS_UDP                  | 14498           | 3000           
DDoS_ICMP                 | 13306           | 3000           
DDoS_HTTP                 | 10560           | 3000           
SQL_injection             | 10297           | 3000           
Uploading                 | 10260           | 3000           
Ransomware                | 10196           | 3000           
Vulnerability_scanner     | 10065           | 3000           
Backdoor                  | 9992            | 3000           
Password                  | 9980            | 3000           
Port_Scanning             | 9690            | 3000           
XSS                       | 9620            | 3000           
DDoS_TCP                  | 6011            | 3000           
Fingerprinting            | 932             | 932            
MITM          

In [20]:
from imblearn.under_sampling import RandomUnderSampler

rus = RandomUnderSampler(sampling_strategy=sampling_strategy, random_state=SEED)
df_no_label, y_str = rus.fit_resample(df_no_label, y_str)

print(df_no_label.shape)

(40299, 58)


In [21]:
# X/y split
X     = df_no_label.drop(columns=[target])
y     = df_no_label[target]

## 1. Pre-processing

In [22]:
# Select only numeric

print("--- Non-numeric cols to drop ---\n\n", X.select_dtypes(include=['str', 'object', 'category']).columns)

X = X.select_dtypes(include=['number'])

print("\n\nRemaining categorical cols:", len(X.select_dtypes(include=['str', 'object', 'category']).columns))

--- Non-numeric cols to drop ---

 Index(['http.file_data', 'http.request.uri.query', 'http.referer',
       'http.request.version', 'mqtt.msg', 'proto', 'http.request.path'],
      dtype='str')


Remaining categorical cols: 0


In [23]:
print(f"NaN values in target variable: {y.isna().sum()}")
print(f"NaN values in features: {X.isna().sum().sum()}")

NaN values in target variable: 0
NaN values in features: 0


In [24]:
X_train, X_test, y_train, y_test, y_str_train, y_str_test = train_test_split(
    X, y, y_str, test_size=0.2, random_state=SEED, stratify=y_str
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

X_train shape: (32239, 50)
X_test shape: (8060, 50)


In [25]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model_results = {}

## 2. LazyPredict
[Docs](https://pypi.org/project/lazypredict/)

In [26]:
from lazypredict.Supervised import LazyClassifier

# With categorical encoding, timeout, cross-validation, and GPU
clf = LazyClassifier(
    verbose=1,                          # Show progress
    ignore_warnings=True,               # Suppress warnings
    custom_metric=None,                 # Use default metrics
    predictions=False,                  # Don't Return predictions
    classifiers='all',                  # Use all available classifiers
    timeout=60,                         # Max time per model in seconds
    cv=5,                               # Cross-validation folds (optional)
)

models, _ = clf.fit(X_train, X_test, y_train, y_test)
print("\n--- Models Evaluated ---")

  0%|          | 0/33 [00:00<?, ?it/s]

/home/uo294319/ML-NIDS-IIoT/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:71: FutureWarning: Class PassiveAggressiveClassifier is deprecated; this is deprecated in version 1.8 and will be removed in 1.10. Use `SGDClassifier(loss='hinge', penalty=None, learning_rate='pa1', eta0=1.0)` instead.
  warnings.warn(msg, category=FutureWarning)
/home/uo294319/ML-NIDS-IIoT/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:71: FutureWarning: Class PassiveAggressiveClassifier is deprecated; this is deprecated in version 1.8 and will be removed in 1.10. Use `SGDClassifier(loss='hinge', penalty=None, learning_rate='pa1', eta0=1.0)` instead.
  warnings.warn(msg, category=FutureWarning)
/home/uo294319/ML-NIDS-IIoT/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:71: FutureWarning: Class PassiveAggressiveClassifier is deprecated; this is deprecated in version 1.8 and will be removed in 1.10. Use `SGDClassifier(loss='hinge', penalty=None, learning_rate='pa1


--- Models Evaluated ---


In [27]:
display(models)

,Accuracy,Balanced Accuracy,ROC AUC,F1 Score,Precision,Recall,Accuracy CV Mean,Accuracy CV Std,Balanced Accuracy CV Mean,Balanced Accuracy CV Std,ROC AUC CV Mean,ROC AUC CV Std,F1 Score CV Mean,F1 Score CV Std,Precision CV Mean,Precision CV Std,Recall CV Mean,Recall CV Std,Time Taken
Model,,,,,,,,,,,,,,,,,,,
ExtraTreesClassifier,0.999256,0.996533,0.999107,0.999255,0.999255,0.999256,0.998976,0.000304,0.995232,0.002285,0.999321,0.000533,0.998975,0.000305,0.998976,0.000304,0.998976,0.000304,1.380869
LGBMClassifier,0.999256,0.995000,0.999660,0.999254,0.999256,0.999256,0.998945,0.000267,0.995407,0.002188,0.999794,0.000364,0.998944,0.000269,0.998945,0.000268,0.998945,0.000267,746.933564
BaggingClassifier,0.997767,0.994196,0.996620,0.997772,0.997781,0.997767,0.998139,0.000260,0.993439,0.001644,0.997520,0.000872,0.998139,0.000257,0.998143,0.000254,0.998139,0.000260,2.757742
XGBClassifier,0.999007,0.994100,0.999342,0.999005,0.999007,0.999007,0.998976,0.000210,0.995424,0.002482,0.999667,0.000610,0.998975,0.000212,0.998977,0.000212,0.998976,0.000210,3.528781
RandomForestClassifier,0.998883,0.994033,0.999663,0.998881,0.998882,0.998883,0.998852,0.000186,0.994591,0.001428,0.999892,0.000071,0.998851,0.000187,0.998851,0.000186,0.998852,0.000186,4.314203
CatBoostClassifier,0.999007,0.993333,0.999867,0.999004,0.999009,0.999007,0.998759,0.000404,0.993008,0.003156,0.999832,0.000281,0.998755,0.000408,0.998759,0.000404,0.998759,0.000404,16.201710
DecisionTreeClassifier,0.997395,0.992462,0.992462,0.997400,0.997407,0.997395,0.997363,0.000340,0.990146,0.002769,0.990146,0.002769,0.997363,0.000341,0.997366,0.000342,0.997363,0.000340,2.315420
ExtraTreeClassifier,0.997519,0.991763,0.991763,0.997521,0.997523,0.997519,0.996960,0.000744,0.989928,0.002166,0.989928,0.002166,0.996964,0.000740,0.996970,0.000735,0.996960,0.000744,1.601887
AdaBoostClassifier,0.987221,0.926428,0.995947,0.986849,0.987056,0.987221,0.986662,0.002096,0.931682,0.006077,0.996104,0.001167,0.986388,0.002110,0.986427,0.002203,0.986662,0.002096,3.222864


In [28]:
display(models.sort_values(by='F1 Score CV Mean', ascending=False).head(3))

,Accuracy,Balanced Accuracy,ROC AUC,F1 Score,Precision,Recall,Accuracy CV Mean,Accuracy CV Std,Balanced Accuracy CV Mean,Balanced Accuracy CV Std,ROC AUC CV Mean,ROC AUC CV Std,F1 Score CV Mean,F1 Score CV Std,Precision CV Mean,Precision CV Std,Recall CV Mean,Recall CV Std,Time Taken
Model,,,,,,,,,,,,,,,,,,,
XGBClassifier,0.999007,0.994100,0.999342,0.999005,0.999007,0.999007,0.998976,0.000210,0.995424,0.002482,0.999667,0.000610,0.998975,0.000212,0.998977,0.000212,0.998976,0.000210,3.528781
ExtraTreesClassifier,0.999256,0.996533,0.999107,0.999255,0.999255,0.999256,0.998976,0.000304,0.995232,0.002285,0.999321,0.000533,0.998975,0.000305,0.998976,0.000304,0.998976,0.000304,1.380869
LGBMClassifier,0.999256,0.995000,0.999660,0.999254,0.999256,0.999256,0.998945,0.000267,0.995407,0.002188,0.999794,0.000364,0.998944,0.000269,0.998945,0.000268,0.998945,0.000267,746.933564
